## Preprocessing

### Imports and Setup

In [1]:
# Imports
import sys
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Add local helpers folder for notebook imports
sys.path.append(str(Path('./helpers').resolve()))

# Import all helper modules
from helpers import *
# from indicators import *
# from time_series_analysis import *
from time_split import *
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', module='scipy', message="^internal gelsd")

In [2]:
# Load the dataset
crypto_data = load_data_csv_files('../data/processed/')

# Remove summary dataframe from the loaded dictionary
crypto_data.pop('top_15_ranking_summary', None)

print(f"\nTotal records loaded: {len(crypto_data)}")


Loaded: avalanche_processed.csv
Loaded: axie_infinity_processed.csv
Loaded: binance_coin_processed.csv
Loaded: bitcoin_processed.csv
Loaded: chainlink_processed.csv
Loaded: ethereum_processed.csv
Loaded: fantom_processed.csv
Loaded: injective_processed.csv
Loaded: litecoin_processed.csv
Loaded: maker_processed.csv
Loaded: render_processed.csv
Loaded: solana_processed.csv
Loaded: the_graph_processed.csv
Loaded: toncoin_processed.csv
Loaded: top_15_ranking_summary.csv
Loaded: tron_processed.csv

Total records loaded: 15


In [17]:
# Load top 15 Data
top_15_summary = pd.read_csv('../data/processed/top_15_ranking_summary.csv')
top_15_summary.head()

,Unnamed: 0,Average_Rank,Sharpe_Ratio,Trend_Slope,Win_Rate,Mean_Volatility,Volatility_Cat
0,bitcoin_cleaned,4.000000,0.054428,22.253055,52.664729,3.192606,Medium Risk
1,the_graph_cleaned,4.333333,0.096246,-0.011538,53.149002,32.215972,High Risk
2,binance_coin_cleaned,4.666667,0.064772,0.277626,52.131588,4.201455,Medium Risk
3,ethereum_cleaned,8.666667,0.039912,1.149978,51.225243,4.185317,Medium Risk
4,solana_cleaned,9.000000,0.068669,0.082410,49.856870,5.834456,High Risk


In [4]:
# Check Data
for name, df in crypto_data.items():
    print(f"\n{name} - Shape: {df.shape}")
    #print(df.head())


avalanche_processed - Shape: (1934, 20)

axie_infinity_processed - Shape: (1889, 20)

binance_coin_processed - Shape: (2980, 20)

bitcoin_processed - Shape: (4129, 20)

chainlink_processed - Shape: (2980, 20)

ethereum_processed - Shape: (2980, 20)

fantom_processed - Shape: (2268, 20)

injective_processed - Shape: (1903, 20)

litecoin_processed - Shape: (4129, 20)

maker_processed - Shape: (2969, 20)

render_processed - Shape: (1502, 20)

solana_processed - Shape: (2097, 20)

the_graph_processed - Shape: (652, 20)

toncoin_processed - Shape: (1593, 20)

tron_processed - Shape: (2980, 20)


### Basic Info 
**This is a quick run down of how to proceed with the preprocessing**
- Keep it simple, I well be looking at bitcoin as my main frame of reference and the copy the process for the other 14 coins. 
- Train Test Split, 80% 20% Split for each coin. 
- Do to being time data we want to pull the 20% from the newest data not oldest data. 
- Already did a lot of feature creation in the last section of this project.
- Need to make sure seasonality and trend are something that the model can use to find some trends.

In [5]:
crypto_data['bitcoin_processed'].head()

,Date,Close,High,Low,Open,Volume,SMA_7,SMA_20,SMA_50,EMA_7,EMA_20,EMA_50,RSI_14,MACD,MACD_Signal,MACD_Histogram,Daily_Return,Weekly_Return,Monthly_Return,Volatility_30
0,2014-09-17,457.334015,468.174011,452.421997,465.864014,21056800,457.334015,457.334015,457.334015,457.334015,457.334015,457.334015,30.165585,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0
1,2014-09-18,424.440002,456.859985,413.104004,456.859985,34483200,424.440002,424.440002,424.440002,449.110512,454.201252,456.044054,30.165585,-2.624024,-0.524805,-2.099219,-7.192558,0.0,0.0,0.0
2,2014-09-19,394.795990,427.834991,384.532013,424.102997,37919700,394.795990,394.795990,394.795990,435.531881,448.543608,453.642169,30.165585,-7.014744,-1.822793,-5.191951,-6.984265,0.0,0.0,0.0
3,2014-09-20,408.903992,423.295990,389.882996,394.673004,36863600,408.903992,408.903992,408.903992,428.874909,444.768406,451.887730,30.165585,-9.249402,-3.308115,-5.941288,3.573492,0.0,0.0,0.0
4,2014-09-21,398.821014,412.425995,393.181000,408.084991,26580100,398.821014,398.821014,398.821014,421.361435,440.392464,449.806683,30.165585,-11.699137,-4.986319,-6.712818,-2.465854,0.0,0.0,0.0


### Feature Engineering  
- Trend
- Day_of_Week
- Month

#### Trend Feature

I made a `Trend` column by checking if `Close > SMA_20`.
- `Up` = above SMA_20
- `Down` = below SMA_20

This is a category, not a number, so I one-hot encoded it.
I did bitcoin first, then used the same process for all 15 coins.

In [6]:
# Trend feature: bitcoin reference
btc = crypto_data['bitcoin_processed'].copy()

# Derive categorical label
btc['Trend'] = np.where(btc['Close'] > btc['SMA_20'], 'Up', 'Down')

# One-hot encode (drop_first=True to avoid dummy variable trap)
btc = pd.get_dummies(btc, columns=['Trend'], drop_first=True)

print("Bitcoin columns after encoding:")
print([c for c in btc.columns if 'Trend' in c])
btc[['Close', 'SMA_20', 'Trend_Up']].head()

Bitcoin columns after encoding:
['Trend_Up']


,Close,SMA_20,Trend_Up
0,457.334015,457.334015,False
1,424.440002,424.440002,False
2,394.795990,394.795990,False
3,408.903992,408.903992,False
4,398.821014,398.821014,False


In [7]:
# Apply Trend encoding to all 15 coins
for name, df in crypto_data.items():
    df['Trend'] = np.where(df['Close'] > df['SMA_20'], 'Up', 'Down')
    crypto_data[name] = pd.get_dummies(df, columns=['Trend'], drop_first=True)

# Verify Trend_Up was added to every coin
missing = [name for name, df in crypto_data.items() if 'Trend_Up' not in df.columns]

if missing:
    print(f"WARNING: Trend_Up missing from: {missing}")
else:
    print("PASSED: Trend_Up column successfully added to all coins.")
    print(f"\nSample — bitcoin_processed Trend_Up value counts:")
    print(crypto_data['bitcoin_processed']['Trend_Up'].value_counts())

PASSED: Trend_Up column successfully added to all coins.

Sample — bitcoin_processed Trend_Up value counts:
Trend_Up
True     2239
False    1890
Name: count, dtype: int64


#### Seasonality Features - Day of Week and Month

I pulled `Day_of_Week` and `Month` from `Date` so the model can learn seasonal patterns.

These are categories, so I used one-hot encoding (not label encoding).
I used `drop_first=True` to avoid duplicate dummy columns.

Same approach: bitcoin first, then all 15 coins.

In [8]:
# Seasonality features: bitcoin reference
btc = crypto_data['bitcoin_processed'].copy()
btc['Date'] = pd.to_datetime(btc['Date'])

# Extract categorical features
btc['Day_of_Week'] = btc['Date'].dt.day_name()
btc['Month'] = btc['Date'].dt.month_name()

# One-hot encode both (drop_first=True to avoid dummy variable trap)
btc = pd.get_dummies(btc, columns=['Day_of_Week', 'Month'], drop_first=True)

dow_cols = [c for c in btc.columns if 'Day_of_Week' in c]
month_cols = [c for c in btc.columns if 'Month' in c]

print(f"Day_of_Week columns ({len(dow_cols)}): {dow_cols}")
print(f"\nMonth columns ({len(month_cols)}): {month_cols}")

Day_of_Week columns (6): ['Day_of_Week_Monday', 'Day_of_Week_Saturday', 'Day_of_Week_Sunday', 'Day_of_Week_Thursday', 'Day_of_Week_Tuesday', 'Day_of_Week_Wednesday']

Month columns (12): ['Monthly_Return', 'Month_August', 'Month_December', 'Month_February', 'Month_January', 'Month_July', 'Month_June', 'Month_March', 'Month_May', 'Month_November', 'Month_October', 'Month_September']


In [9]:
# Apply Day_of_Week and Month encoding to all 15 coins
for name, df in crypto_data.items():
    df['Date'] = pd.to_datetime(df['Date'])
    df['Day_of_Week'] = df['Date'].dt.day_name()
    df['Month'] = df['Date'].dt.month_name()
    crypto_data[name] = pd.get_dummies(df, columns=['Day_of_Week', 'Month'], drop_first=True)

# Verify both feature groups were added to every coin
missing_dow   = [n for n, df in crypto_data.items() if not any('Day_of_Week' in c for c in df.columns)]
missing_month = [n for n, df in crypto_data.items() if not any('Month' in c for c in df.columns)]

if missing_dow or missing_month:
    print(f"WARNING: Day_of_Week missing from: {missing_dow}")
    print(f"WARNING: Month missing from: {missing_month}")
else:
    print("PASSED: Day_of_Week and Month columns successfully added to all coins.")
    sample_df = crypto_data['bitcoin_processed']
    print(f"\nBitcoin total columns after all encoding: {sample_df.shape[1]}")

PASSED: Day_of_Week and Month columns successfully added to all coins.

Bitcoin total columns after all encoding: 38


### Split Data

I split before scaling to avoid data leakage.

- 80% train = older data
- 20% test = newer data

Since this is time-series data, I used a time split (not random).

In [10]:
split_data = split_dataset_dict_time(crypto_data, test_size=0.2, date_col='Date')
print(f"\nSplit data keys: {list(split_data.keys())}")


Split data keys: ['avalanche_processed', 'axie_infinity_processed', 'binance_coin_processed', 'bitcoin_processed', 'chainlink_processed', 'ethereum_processed', 'fantom_processed', 'injective_processed', 'litecoin_processed', 'maker_processed', 'render_processed', 'solana_processed', 'the_graph_processed', 'toncoin_processed', 'tron_processed']


In [11]:
for key in split_data.keys():
    print(f"{key} - Train shape: {split_data[key]['train'].shape}, Test shape: {split_data[key]['test'].shape}")

avalanche_processed - Train shape: (1547, 38), Test shape: (387, 38)
axie_infinity_processed - Train shape: (1511, 38), Test shape: (378, 38)
binance_coin_processed - Train shape: (2384, 38), Test shape: (596, 38)
bitcoin_processed - Train shape: (3303, 38), Test shape: (826, 38)
chainlink_processed - Train shape: (2384, 38), Test shape: (596, 38)
ethereum_processed - Train shape: (2384, 38), Test shape: (596, 38)
fantom_processed - Train shape: (1814, 38), Test shape: (454, 38)
injective_processed - Train shape: (1522, 38), Test shape: (381, 38)
litecoin_processed - Train shape: (3303, 38), Test shape: (826, 38)
maker_processed - Train shape: (2375, 38), Test shape: (594, 38)
render_processed - Train shape: (1201, 38), Test shape: (301, 38)
solana_processed - Train shape: (1677, 38), Test shape: (420, 38)
the_graph_processed - Train shape: (521, 38), Test shape: (131, 38)
toncoin_processed - Train shape: (1274, 38), Test shape: (319, 38)
tron_processed - Train shape: (2384, 38), Test 

### Scale Standardization

#### Why Standardization Matters

Some columns are much bigger than others (like `Volume`), so models can get biased by scale.

I used `StandardScaler` so features are on a similar range.
- fit on train only
- transform train + test

That keeps things fair and avoids leaking test info.

I did not scale:
- `Date`
- one-hot columns (`Trend_Up`, `Day_of_Week_*`, `Month_*`)

In [12]:
# Scale Standardization: bitcoin reference
btc_train = split_data['bitcoin_processed']['train'].copy()
btc_test  = split_data['bitcoin_processed']['test'].copy()

# Identify columns to scale: numeric only, exclude Date and one-hot encoded columns
exclude_prefixes = ('Day_of_Week_', 'Month_', 'Trend_')
exclude_cols     = {'Date'}

scale_cols = [
    c for c in btc_train.columns
    if c not in exclude_cols
    and not any(c.startswith(p) for p in exclude_prefixes)
    and pd.api.types.is_numeric_dtype(btc_train[c])
]

print(f"Columns to scale ({len(scale_cols)}):")
print(scale_cols)

Columns to scale (19):
['Close', 'High', 'Low', 'Open', 'Volume', 'SMA_7', 'SMA_20', 'SMA_50', 'EMA_7', 'EMA_20', 'EMA_50', 'RSI_14', 'MACD', 'MACD_Signal', 'MACD_Histogram', 'Daily_Return', 'Weekly_Return', 'Monthly_Return', 'Volatility_30']


In [13]:
# Fit scaler on training data only, then transform train and test
scaler = StandardScaler()

btc_train[scale_cols] = scaler.fit_transform(btc_train[scale_cols])
btc_test[scale_cols]  = scaler.transform(btc_test[scale_cols])

print("Bitcoin — scaled train stats (should be ~mean=0, std=1):")
print(btc_train[scale_cols].describe().loc[['mean', 'std']].round(4))

Bitcoin — scaled train stats (should be ~mean=0, std=1):
       Close    High     Low    Open  Volume   SMA_7  SMA_20  SMA_50   EMA_7  \
mean -0.0000  0.0000  0.0000  0.0000  0.0000  0.0000  0.0000 -0.0000 -0.0000   
std   1.0002  1.0002  1.0002  1.0002  1.0002  1.0002  1.0002  1.0002  1.0002   

      EMA_20  EMA_50  RSI_14    MACD  MACD_Signal  MACD_Histogram  \
mean  0.0000  0.0000 -0.0000 -0.0000      -0.0000         -0.0000   
std   1.0002  1.0002  1.0002  1.0002       1.0002          1.0002   

      Daily_Return  Weekly_Return  Monthly_Return  Volatility_30  
mean       -0.0000        -0.0000         -0.0000        -0.0000  
std         1.0002         1.0002          1.0002         1.0002  


In [14]:
# Apply scaling to all 15 coins
scalers = {}  # store each coin's fitted scaler for later use

for name in split_data.keys():
    train_df = split_data[name]['train'].copy()
    test_df  = split_data[name]['test'].copy()

    # Identify scale columns for this coin (same logic as above)
    cols = [
        c for c in train_df.columns
        if c not in exclude_cols
        and not any(c.startswith(p) for p in exclude_prefixes)
        and pd.api.types.is_numeric_dtype(train_df[c])
    ]

    sc = StandardScaler()
    train_df[cols] = sc.fit_transform(train_df[cols])
    test_df[cols]  = sc.transform(test_df[cols])

    scalers[name] = sc
    split_data[name]['train'] = train_df
    split_data[name]['test']  = test_df

# Verify: check all scalers were stored
missing_scalers = [n for n in split_data.keys() if n not in scalers]

if missing_scalers:
    print(f"WARNING: Scaler missing for: {missing_scalers}")
else:
    print(f"PASSED: Scalers fitted and applied to all {len(scalers)} coins.")
    print(f"\nSample — bitcoin_processed train mean (should be ~0):")
    print(split_data['bitcoin_processed']['train'][scale_cols].mean().round(4))

PASSED: Scalers fitted and applied to all 15 coins.

Sample — bitcoin_processed train mean (should be ~0):
Close            -0.0
High              0.0
Low               0.0
Open              0.0
Volume            0.0
SMA_7             0.0
SMA_20            0.0
SMA_50           -0.0
EMA_7            -0.0
EMA_20            0.0
EMA_50            0.0
RSI_14           -0.0
MACD             -0.0
MACD_Signal      -0.0
MACD_Histogram   -0.0
Daily_Return     -0.0
Weekly_Return    -0.0
Monthly_Return   -0.0
Volatility_30    -0.0
dtype: float64


### Spot-Check (Before Save)

Quick check on bitcoin train/test after encoding, splitting, and scaling.
- `head()` to check structure
- `describe()` to check numeric ranges

In [15]:
# Spot-check bitcoin train/test sets
btc_train_check = split_data['bitcoin_processed']['train']
btc_test_check  = split_data['bitcoin_processed']['test']

print("Bitcoin train head:")
display(btc_train_check.head())

print("\nBitcoin test head:")
display(btc_test_check.head())

print("\nBitcoin train describe (scaled numeric columns):")
display(btc_train_check[scale_cols].describe().round(4))

print("\nBitcoin test describe (scaled numeric columns):")
display(btc_test_check[scale_cols].describe().round(4))

Bitcoin train head:


,Date,Close,High,Low,Open,Volume,SMA_7,SMA_20,SMA_50,EMA_7,...,Month_December,Month_February,Month_January,Month_July,Month_June,Month_March,Month_May,Month_November,Month_October,Month_September
0,2014-09-17,-0.847397,-0.846173,-0.848077,-0.846268,-0.852940,-0.846995,-0.846022,-0.844083,-0.847621,...,False,False,False,False,False,False,False,False,False,True
1,2014-09-18,-0.849457,-0.846864,-0.850610,-0.846832,-0.852244,-0.849058,-0.848090,-0.846166,-0.848137,...,False,False,False,False,False,False,False,False,False,True
2,2014-09-19,-0.851314,-0.848639,-0.852451,-0.848883,-0.852066,-0.850918,-0.849955,-0.848042,-0.848989,...,False,False,False,False,False,False,False,False,False,True
3,2014-09-20,-0.850430,-0.848916,-0.852106,-0.850726,-0.852121,-0.850033,-0.849067,-0.847149,-0.849407,...,False,False,False,False,False,False,False,False,False,True
4,2014-09-21,-0.851062,-0.849581,-0.851894,-0.849887,-0.852654,-0.850665,-0.849701,-0.847787,-0.849879,...,False,False,False,False,False,False,False,False,False,True



Bitcoin test head:


,Date,Close,High,Low,Open,Volume,SMA_7,SMA_20,SMA_50,EMA_7,...,Month_December,Month_February,Month_January,Month_July,Month_June,Month_March,Month_May,Month_November,Month_October,Month_September
3303,2023-10-03,0.842060,0.816405,0.876149,0.847228,-0.262609,0.828489,0.810773,0.804326,0.832790,...,False,False,False,False,False,False,False,False,True,False
3304,2023-10-04,0.865198,0.826152,0.878217,0.842270,-0.276319,0.841451,0.814734,0.802590,0.841719,...,False,False,False,False,False,False,False,False,True,False
3305,2023-10-05,0.841179,0.842363,0.886431,0.865414,-0.238271,0.844985,0.817272,0.800962,0.842398,...,False,False,False,False,False,False,False,False,True,False
3306,2023-10-06,0.874419,0.852185,0.876120,0.841208,-0.154537,0.854257,0.821606,0.802586,0.851234,...,False,False,False,False,False,False,False,False,True,False
3307,2023-10-07,0.875812,0.838465,0.918309,0.874690,-0.514298,0.863225,0.826117,0.805015,0.858211,...,False,False,False,False,False,False,False,False,True,False



Bitcoin train describe (scaled numeric columns):


,Close,High,Low,Open,Volume,SMA_7,SMA_20,SMA_50,EMA_7,EMA_20,EMA_50,RSI_14,MACD,MACD_Signal,MACD_Histogram,Daily_Return,Weekly_Return,Monthly_Return,Volatility_30
count,3303.0000,3303.0000,3303.0000,3303.0000,3303.0000,3303.0000,3303.0000,3303.0000,3303.0000,3303.0000,3303.0000,3303.0000,3303.0000,3303.0000,3303.0000,3303.0000,3303.0000,3303.0000,3303.0000
mean,-0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,-0.0000,-0.0000,0.0000,0.0000,-0.0000,-0.0000,-0.0000,-0.0000,-0.0000,-0.0000,-0.0000,-0.0000
std,1.0002,1.0002,1.0002,1.0002,1.0002,1.0002,1.0002,1.0002,1.0002,1.0002,1.0002,1.0002,1.0002,1.0002,1.0002,1.0002,1.0002,1.0002,1.0002
min,-0.8649,-0.8618,-0.8662,-0.8644,-0.8537,-0.8628,-0.8605,-0.8583,-0.8628,-0.8624,-0.8642,-3.0326,-5.4034,-5.0003,-6.4589,-10.0225,-4.5913,-2.7047,-2.1577
25%,-0.8261,-0.8251,-0.8266,-0.8259,-0.8469,-0.8260,-0.8258,-0.8257,-0.8264,-0.8277,-0.8315,-0.7043,-0.1788,-0.1898,-0.1408,-0.3963,-0.5066,-0.6318,-0.6941
50%,-0.3753,-0.3737,-0.3749,-0.3746,-0.3004,-0.3730,-0.3749,-0.3611,-0.3755,-0.3752,-0.3676,-0.0826,-0.0553,-0.0585,0.0024,-0.0173,-0.0724,-0.1572,-0.1150
75%,0.4903,0.4958,0.4990,0.4894,0.5532,0.5032,0.5032,0.5082,0.5093,0.5108,0.5317,0.6417,0.1688,0.1790,0.1903,0.4091,0.4534,0.4741,0.5468
max,3.3561,3.3301,3.3994,3.3548,17.3414,3.2331,3.0966,2.9756,3.1867,3.1073,2.8918,2.9343,5.5195,4.9641,4.9750,6.7202,7.2364,7.5333,3.7046



Bitcoin test describe (scaled numeric columns):


,Close,High,Low,Open,Volume,SMA_7,SMA_20,SMA_50,EMA_7,EMA_20,EMA_50,RSI_14,MACD,MACD_Signal,MACD_Histogram,Daily_Return,Weekly_Return,Monthly_Return,Volatility_30
count,826.0000,826.0000,826.0000,826.0000,826.0000,826.0000,826.0000,826.0000,826.0000,826.0000,826.0000,826.0000,826.0000,826.0000,826.0000,826.0000,826.0000,826.0000,826.0000
mean,4.0515,4.0135,4.0957,4.0461,1.3398,4.0438,4.0275,3.9891,4.0469,4.0364,4.0138,0.1133,0.5033,0.5326,-0.0107,-0.0039,-0.0167,-0.0426,-0.6165
std,1.6089,1.5897,1.6356,1.6121,1.2640,1.6172,1.6377,1.6814,1.6164,1.6343,1.6688,0.9032,2.1866,2.1845,2.1952,0.6682,0.6317,0.5946,0.4293
min,0.7999,0.7708,0.8338,0.7999,-0.5747,0.8221,0.8108,0.8010,0.8236,0.8156,0.8405,-2.1016,-6.3280,-5.8349,-5.8273,-2.3810,-2.0272,-1.2230,-1.4393
25%,2.9315,2.9231,2.9583,2.9298,0.4133,2.9627,2.9194,2.9684,2.9577,2.9663,3.0307,-0.5599,-1.0068,-1.0030,-1.4604,-0.3446,-0.4197,-0.4396,-0.9264
50%,4.2858,4.2531,4.1927,4.2410,1.0748,4.2547,4.1534,3.6304,4.2868,4.2944,3.8523,0.0456,0.4835,0.5439,-0.0979,-0.0312,-0.0607,-0.1424,-0.6483
75%,5.4729,5.4598,5.5126,5.4721,1.9932,5.4763,5.4011,5.3818,5.4569,5.3975,5.3861,0.7110,1.7040,1.6868,1.5060,0.3015,0.2900,0.2392,-0.3739
max,6.9379,6.8392,7.0596,6.9370,8.5684,6.8251,6.5657,6.5106,6.7643,6.5961,6.5191,2.4795,7.3973,7.0899,7.1763,3.2055,2.8950,2.1209,0.6852


### Clean top_15_summary data for modeling step
This well be used for comparison not modeling.

In [20]:
# Clean top_15_summary for modeling
unnamed_cols = ['Unnamed: 0']

if unnamed_cols:
    top_15_summary = top_15_summary.rename(columns={unnamed_cols[0]: 'Coin'})

if 'Coin' in top_15_summary.columns:
    top_15_summary['Coin'] = top_15_summary['Coin'].astype(str).str.replace('_cleaned', '', regex=False)

top_15_summary.head()

,Coin,Average_Rank,Sharpe_Ratio,Trend_Slope,Win_Rate,Mean_Volatility,Volatility_Cat
0,bitcoin,4.000000,0.054428,22.253055,52.664729,3.192606,Medium Risk
1,the_graph,4.333333,0.096246,-0.011538,53.149002,32.215972,High Risk
2,binance_coin,4.666667,0.064772,0.277626,52.131588,4.201455,Medium Risk
3,ethereum,8.666667,0.039912,1.149978,51.225243,4.185317,Medium Risk
4,solana,9.000000,0.068669,0.082410,49.856870,5.834456,High Risk


### Save Model Data

In [21]:
# Save train/test model data + unscaled graph data
train_dir = Path('../data/model_data/Train')
test_dir = Path('../data/model_data/Test')
graph_dir = Path('../data/model_data/graph')

train_dir.mkdir(parents=True, exist_ok=True)
test_dir.mkdir(parents=True, exist_ok=True)
graph_dir.mkdir(parents=True, exist_ok=True)

# Save scaled train/test sets
for name in split_data.keys():
    clean_name = name.replace('_processed', '')
    split_data[name]['train'].to_csv(train_dir / f"{clean_name}_train.csv", index=False)
    split_data[name]['test'].to_csv(test_dir / f"{clean_name}_test.csv", index=False)

# Save unscaled data for graphing
for name, df in crypto_data.items():
    clean_name = name.replace('_processed', '')
    df.to_csv(graph_dir / f"{clean_name}_graph.csv", index=False)

# Save cleaned top 15 summary for graphing / reference
top_15_summary.to_csv(graph_dir / 'top_15_summary_graph.csv', index=False)

# Verify files were written
train_files = list(train_dir.glob('*.csv'))
test_files = list(test_dir.glob('*.csv'))
graph_files = list(graph_dir.glob('*.csv'))

expected_train_test = len(split_data)
expected_graph = len(crypto_data) + 1

if len(train_files) == expected_train_test and len(test_files) == expected_train_test and len(graph_files) == expected_graph:
    print(f"PASSED: {len(train_files)} train, {len(test_files)} test, and {len(graph_files)} graph files saved.")
    print(f"\nTrain: {train_dir.resolve()}")
    print(f"Test:  {test_dir.resolve()}")
    print(f"Graph: {graph_dir.resolve()}")
else:
    print(f"WARNING: Expected {expected_train_test} train, {expected_train_test} test, and {expected_graph} graph files.")
    print(f"  Train files found: {len(train_files)}")
    print(f"  Test files found:  {len(test_files)}")
    print(f"  Graph files found: {len(graph_files)}")

PASSED: 15 train, 15 test, and 16 graph files saved.

Train: C:\Users\tanne\OneDrive\Documents\Code\Top50CyrptoAnaliyst\data\model_data\Train
Test:  C:\Users\tanne\OneDrive\Documents\Code\Top50CyrptoAnaliyst\data\model_data\Test
Graph: C:\Users\tanne\OneDrive\Documents\Code\Top50CyrptoAnaliyst\data\model_data\Graph


### Summary

This notebook prepared the 15 cryptocurrency datasets for model training. The following steps were completed in order:

| Step | What was done | Why |
|---|---|---|
| **1. Load & Clean** | Loaded all 15 processed CSVs into `crypto_data`; dropped `top_15_ranking_summary` | Kept only the coin datasets for modeling |
| **2. Feature Engineering** | Derived `Trend` (Close vs SMA_20), `Day_of_Week`, and `Month`; one-hot encoded all three with `drop_first=True` | Added trend + seasonality signals without false numeric ordering |
| **3. Train / Test Split** | 80% train (oldest data), 20% test (newest data) using `split_dataset_dict_time()` | Time-series split prevents future leakage into training |
| **4. Scale Standardization** | Fit `StandardScaler` on train only, then transformed train and test | Put numeric features on similar scale while avoiding leakage |
| **5. Spot-Check** | Ran `head()` and `describe()` on bitcoin train/test after preprocessing | Quick validation that structure and scaled stats looked correct |
| **6. Save Outputs** | Saved scaled train/test files to `data/model_data/Train` and `data/model_data/Test`; saved unscaled graph files to `data/model_data/graph` | Produced model-ready datasets plus chart-friendly unscaled data |

**Output:** `split_data` contains scaled `train` and `test` DataFrames per coin, and `scalers` stores each fitted scaler for reuse.